# MaJETstik: analysing the output

A guided tour of `data/<run>_nano.root`, the flat NanoAOD-like file the pipeline produces.

**Before running this notebook**, produce a sample:

```bash
source setup_env.sh
python3 main.py                 # 1000 Z -> ee events, ~3 minutes
jupyter lab examples/analysis_tutorial.ipynb
```

Everything below is plain `uproot` + `awkward` + `numpy` + `matplotlib`.
Nothing from MaJETstik itself is needed to *read* the file — that is the point of the format.


## 1. Open the file

A MaJETstik file holds three objects: the `Events` tree, a `ReadMe` describing every
branch, and a `Metadata` record saying exactly how the file was made.


In [ ]:
import json
from pathlib import Path

import awkward as ak
import numpy as np
import uproot
import matplotlib.pyplot as plt

RUN = 'zee_jets'          # change to 'dijet' if you ran that config
PATH = Path('..') / 'data' / f'{RUN}_nano.root'
assert PATH.is_file(), f'{PATH} not found - run `python3 main.py` first'

f = uproot.open(PATH)
print('objects in the file:', f.keys())
events_tree = f['Events']
print('events:', events_tree.num_entries)


### What was this file made from?

The provenance record travels *inside* the ROOT file, so it can never be separated
from the data it describes.


In [ ]:
meta = json.loads(str(f['Metadata']))
print('seed        :', meta['seed'])
print('pythia card :', Path(meta['cards']['pythia_card']).name)
print('delphes card:', Path(meta['cards']['delphes_card']).name)
print('jets        : {jet_algorithm} R={jet_r}, pT > {jet_pt_min} GeV'.format(**meta['config']))
print()
for k, v in meta['software'].items():
    print(f'  {k:16s} {v}')


In [ ]:
# The branch documentation is embedded too.
print(str(f['ReadMe'])[:1200], '...')


## 2. Load the branches

Branches follow the rule `<Collection>_<variable>`, and all branches sharing a prefix
have the same length within an event.


In [ ]:
ev = events_tree.arrays([
    'Jet_pt', 'Jet_eta', 'Jet_phi', 'Jet_mass', 'Jet_nConstituents',
    'Jet_tau1', 'Jet_tau2', 'Jet_tau3', 'Jet_softdrop_mass',
    'Jet_btag', 'Jet_flavor', 'Jet_isLepton', 'Jet_genJetIdx',
    'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_charge', 'Electron_iso',
    'Muon_pt', 'MET_pt', 'MET_phi', 'GenJet_pt',
])

print('jets in event 0 :', ak.to_list(ev['Jet_pt'][0]))
print('electrons       :', ak.to_list(ev['Electron_pt'][0]))
print('MET             :', float(ev['MET_pt'][0]), 'GeV')


## 3. The one gotcha: some "jets" are electrons

Particle flow reconstructs electrons and then hands them to the jet algorithm along
with everything else. So a $Z \to e^+e^-$ event genuinely produces two jets that *are*
the electrons — you can see it in the jet's constituent count and near-zero mass.

This is not a bug; real experiments remove them with *overlap removal*. Stage 4 flags
them in `Jet_isLepton`, and **a hadronic analysis should always cut on it**.


In [ ]:
lepton_jets   = ev['Jet_isLepton'] == 1
hadronic_jets = ev['Jet_isLepton'] == 0

n_lep = int(ak.sum(ak.num(ev['Jet_pt'][lepton_jets])))
n_had = int(ak.sum(ak.num(ev['Jet_pt'][hadronic_jets])))
print(f'lepton-like jets : {n_lep}')
print(f'hadronic jets    : {n_had}')
print()
print('mean constituents, lepton-like jets:',
      float(ak.mean(ak.flatten(ev['Jet_nConstituents'][lepton_jets]))))
print('mean constituents, hadronic jets   :',
      float(ak.mean(ak.flatten(ev['Jet_nConstituents'][hadronic_jets]))))


## 4. The jet transverse-momentum spectrum

Steeply falling, as QCD radiation always is: the probability of radiating a gluon
falls roughly as $1/p_T$, so the spectrum is close to a power law.


In [ ]:
jet_pt = np.asarray(ak.flatten(ev['Jet_pt'][hadronic_jets]))
gen_pt = np.asarray(ak.flatten(ev['GenJet_pt']))

bins = np.linspace(20, max(np.percentile(jet_pt, 99.5), 60), 40)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.hist(gen_pt,  bins=bins, histtype='step', color='#eb6834', label='generator jets')
ax.hist(jet_pt,  bins=bins, histtype='step', color='#2a78d6', label='reconstructed jets')
ax.set_yscale('log')
ax.set_xlabel(r'jet $p_T$  [GeV]'); ax.set_ylabel('jets / bin')
ax.set_title('Jet transverse-momentum spectrum', loc='left')
ax.legend(frameon=False); ax.grid(alpha=0.3)
plt.show()


## 5. The Z peak — is the chain working?

The invariant mass of the two electrons should peak at the PDG Z mass, 91.19 GeV.
This is the single best end-to-end check of the whole pipeline.

The peak sits slightly *below* 91.19 and has a low-mass tail, both physical: final-state
radiation carries energy away from the electrons, and the detector's energy resolution
smears the rest.


In [ ]:
two_e = ak.num(ev['Electron_pt']) >= 2
pt  = ev['Electron_pt'][two_e][:, :2]
eta = ev['Electron_eta'][two_e][:, :2]
phi = ev['Electron_phi'][two_e][:, :2]
q   = ev['Electron_charge'][two_e][:, :2]

px, py = pt*np.cos(phi), pt*np.sin(phi)
pz = pt*np.sinh(eta)
E  = np.sqrt(px**2 + py**2 + pz**2)      # m_e = 0.511 MeV is negligible here

m2 = ((E[:, 0]+E[:, 1])**2 - (px[:, 0]+px[:, 1])**2
      - (py[:, 0]+py[:, 1])**2 - (pz[:, 0]+pz[:, 1])**2)
mass = np.asarray(np.sqrt(np.maximum(m2, 0)))
mass = mass[np.asarray(q[:, 0]*q[:, 1]) < 0]     # opposite charge

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.hist(mass, bins=np.linspace(60, 120, 40), histtype='step', color='#2a78d6')
ax.axvline(91.1876, color='#eb6834', ls='--', lw=1.5)
ax.set_xlabel(r'$m(e^+e^-)$  [GeV]'); ax.set_ylabel('events / bin')
ax.set_title('Dielectron invariant mass', loc='left'); ax.grid(alpha=0.3)
plt.show()

print(f'{len(mass)} pairs, median m(ee) = {np.median(mass):.2f} GeV  (PDG: 91.19)')


## 6. Substructure: grooming and N-subjettiness

**Soft drop** walks the jet's clustering tree backwards and throws away the softer
branch whenever it carries less than $z_{cut}$ of the pair's momentum. What survives
is the hard core, so the groomed mass is closer to the mass of whatever made the jet.

**$\tau_{21} = \tau_2/\tau_1$** is small when a jet has two distinct prongs — the
signature of a boosted $W$, $Z$ or Higgs decaying to a quark pair.

With the default $R = 0.4$ jets from $Z$+jets, almost every jet is one-pronged and
grooming has little to remove. That *is* the lesson: substructure is a large-radius,
high-$p_T$ tool. Run `python3 main.py --config examples/config_dijet.json` and set
`RUN = 'dijet'` at the top of this notebook to see these plots come alive.


In [ ]:
mass_raw = np.asarray(ak.flatten(ev['Jet_mass'][hadronic_jets]))
mass_sd  = np.asarray(ak.flatten(ev['Jet_softdrop_mass'][hadronic_jets]))
mass_sd  = mass_sd[mass_sd >= 0]           # -1 means grooming rejected the jet

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))

bins = np.linspace(0, max(np.percentile(mass_raw, 99), 10), 40)
axes[0].hist(mass_raw, bins=bins, histtype='step', color='#2a78d6', label='ungroomed')
axes[0].hist(mass_sd,  bins=bins, histtype='step', color='#eb6834', label='soft-drop')
axes[0].set_xlabel('jet mass  [GeV]'); axes[0].set_ylabel('jets / bin')
axes[0].set_title('Grooming', loc='left'); axes[0].legend(frameon=False)

t1 = np.asarray(ak.flatten(ev['Jet_tau1'][hadronic_jets]))
t2 = np.asarray(ak.flatten(ev['Jet_tau2'][hadronic_jets]))
ok = t1 > 0
axes[1].hist(t2[ok]/t1[ok], bins=np.linspace(0, 1.2, 40), histtype='step', color='#2a78d6')
axes[1].set_xlabel(r'$\tau_{21} = \tau_2/\tau_1$'); axes[1].set_ylabel('jets / bin')
axes[1].set_title('N-subjettiness ratio', loc='left')

for a in axes: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 7. Jet energy response — what the detector did

`Jet_genJetIdx` points at the generator-level jet matched within $\Delta R < R/2$,
so the ratio of the two momenta measures how faithfully the detector measured the jet.

Expect a mean slightly below 1: some energy leaks outside the jet cone, and neutrinos
inside the jet are invisible.


In [ ]:
ratios = []
jet_pt_j = ev['Jet_pt'][hadronic_jets]
gidx_j   = ev['Jet_genJetIdx'][hadronic_jets]

for i in range(len(jet_pt_j)):
    gi = np.asarray(gidx_j[i]); g = np.asarray(ev['GenJet_pt'][i])
    sel = gi >= 0
    if sel.sum() and g.size:
        ratios.append(np.asarray(jet_pt_j[i])[sel] / g[gi[sel]])

ratio = np.concatenate(ratios)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.hist(ratio, bins=np.linspace(0.3, 1.7, 40), histtype='step', color='#2a78d6')
ax.axvline(1.0, color='#52514e', ls='--', lw=1.2)
ax.set_xlabel(r'$p_T^{\rm reco} / p_T^{\rm gen}$'); ax.set_ylabel('jets / bin')
ax.set_title('Jet energy response', loc='left'); ax.grid(alpha=0.3)
plt.show()

print(f'mean {ratio.mean():.3f}   RMS {ratio.std():.3f}   ({ratio.size} matched jets)')


## 8. Opening up a jet: the constituent map

A ROOT tree stores only **one** level of variable length per branch, so the
constituents of each jet cannot be a nested array. NanoAOD flattens the relation into
two parallel arrays, one entry per (jet, constituent) pair — exactly what CMS's own
PFNano does.


In [ ]:
cons = events_tree.arrays(['JetConstituent_jetIdx', 'JetConstituent_pfIdx',
                           'PFCand_pt', 'PFCand_eta', 'PFCand_phi', 'PFCand_pdgId'],
                          entry_stop=200)

# Find an event with a well-populated hadronic jet to look inside.
nconst = ev['Jet_nConstituents'][hadronic_jets]
cand_events = [i for i in range(len(cons['PFCand_pt']))
               if len(nconst[i]) and max(nconst[i]) >= 8]
i = cand_events[0]

jet_idx = np.asarray(cons['JetConstituent_jetIdx'][i])
pf_idx  = np.asarray(cons['JetConstituent_pfIdx'][i])
pf_pt   = np.asarray(cons['PFCand_pt'][i])
pf_id   = np.asarray(cons['PFCand_pdgId'][i])

j = int(np.argmax([np.sum(jet_idx == k) for k in range(jet_idx.max()+1)]))
members = pf_idx[jet_idx == j]

print(f'event {i}, jet {j}: {len(members)} constituents, '
      f'scalar sum pT = {pf_pt[members].sum():.1f} GeV')
print()
print(f"{'pdgId':>7} {'pT [GeV]':>10}")
for k in members[np.argsort(-pf_pt[members])][:10]:
    print(f'{int(pf_id[k]):>7} {pf_pt[k]:>10.2f}')

# 22 = photon tower, 130 = neutral-hadron tower, everything else is a track.


## 9. Where to go next

- `docs/theory.md` — the physics behind every stage, with equations and references
- `docs/output_format.md` — every branch, and more on the constituent map
- `docs/extending.md` — recipes: a new process, a new detector, a new observable

Two experiments worth doing right now:

1. **Make substructure matter.** `python3 main.py --config examples/config_dijet.json`,
   then set `RUN = 'dijet'` in cell 1 and re-run. Section 6 changes completely.
2. **Remove the underlying event.** Set `PartonLevel:MPI = off` in
   `generators/cards/zee_jets.cmnd`, re-run with a new `--run-name`, and compare jet
   masses and multiplicities against this sample.


In [ ]:
f.close()
